# Ensemble Sampling v1.2

MVCAF backbone test v1.2 체크포인트를 이용해 backbone 간 representation/예측 보완성을 분석합니다.

In [1]:
from __future__ import annotations

import ast
import json
import os
import sys
from dataclasses import dataclass
from pathlib import Path
from itertools import combinations

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from IPython.display import display

ROOT = Path.cwd().resolve().parent
SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from augmentations import build_default_transforms
from reproducibility import make_generator, seed_everything, seed_worker

DATA_DIR = ROOT / "data"
EVAL_CSV = ROOT / "outputs" / "eda_preprocessing" / "mv_caf_backbone_test" / "v1.2" / "mv_caf_backbone_test_v1.2_eval.csv"
OUT_DIR = ROOT / "outputs" / "eda_preprocessing" / "ensemble_sampling" / "v1.2"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CFG = {
    "IMG_SIZE": 224,
    "BATCH_SIZE": 16,
    "SEED": 42,
    "NUM_WORKERS": 8,
}

seed_everything(CFG["SEED"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("eval_csv:", EVAL_CSV)
print("out_dir:", OUT_DIR)


device: cuda
eval_csv: /media/hdd0/whyz/structure-stability/outputs/eda_preprocessing/mv_caf_backbone_test/v1.2/mv_caf_backbone_test_v1.2_eval.csv
out_dir: /media/hdd0/whyz/structure-stability/outputs/eda_preprocessing/ensemble_sampling/v1.2


In [2]:
class MultiViewDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.label_map = {"stable": 0, "unstable": 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sample_id = str(row["id"])
        base_dir = self.root_dir
        if ("sample_dir" in self.df.columns) and pd.notna(row.get("sample_dir", np.nan)):
            base_dir = str(row["sample_dir"])
        folder_path = os.path.join(base_dir, sample_id)

        views = []
        for name in ["front", "top"]:
            img_path = os.path.join(folder_path, f"{name}.png")
            image = Image.open(img_path).convert("RGB")
            if self.transform:
                image = self.transform(image)
            views.append(image)

        if self.is_test:
            return views
        return views, self.label_map[row["label"]]


@dataclass(frozen=True)
class FlexibleCAFConfig:
    backbone_name: str = "efficientnetv2_rw_s"
    pretrained: bool = True
    attn_dim: int = 256
    num_heads: int = 8
    num_layers: int = 2
    dropout: float = 0.1
    classifier_hidden_dim: int = 512
    classifier_mid_dim: int = 128
    classifier_dropout: float = 0.3
    out_dim: int = 1


class CrossAttentionBlock(nn.Module):
    def __init__(self, dim: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        self.norm_q = nn.LayerNorm(dim)
        self.norm_kv = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)

    def forward(self, q_tokens: torch.Tensor, kv_tokens: torch.Tensor) -> torch.Tensor:
        q = self.norm_q(q_tokens)
        kv = self.norm_kv(kv_tokens)
        attn_out, _ = self.attn(q, kv, kv, need_weights=False)
        return q_tokens + attn_out


class MultiViewFlexibleCAF(nn.Module):
    def __init__(self, config: FlexibleCAFConfig):
        super().__init__()
        self.config = config
        self.backbone = timm.create_model(
            config.backbone_name,
            pretrained=config.pretrained,
            num_classes=0,
            global_pool="",
        )
        self.feature_dim = getattr(self.backbone, "num_features")
        self.cnn_proj = nn.Conv2d(self.feature_dim, config.attn_dim, kernel_size=1, bias=False)
        self.token_proj = nn.Linear(self.feature_dim, config.attn_dim, bias=False)
        self.cross_12 = nn.ModuleList(
            [CrossAttentionBlock(config.attn_dim, config.num_heads, config.dropout) for _ in range(config.num_layers)]
        )
        self.cross_21 = nn.ModuleList(
            [CrossAttentionBlock(config.attn_dim, config.num_heads, config.dropout) for _ in range(config.num_layers)]
        )
        self.classifier = nn.Sequential(
            nn.Linear(config.attn_dim * 2, config.classifier_hidden_dim),
            nn.BatchNorm1d(config.classifier_hidden_dim),
            nn.ReLU(),
            nn.Dropout(config.classifier_dropout),
            nn.Linear(config.classifier_hidden_dim, config.classifier_mid_dim),
            nn.ReLU(),
            nn.Linear(config.classifier_mid_dim, config.out_dim),
        )

    def _to_tokens(self, feat: torch.Tensor) -> torch.Tensor:
        if feat.ndim == 4:
            if feat.shape[1] == self.feature_dim:
                feat = self.cnn_proj(feat)
                return feat.flatten(2).transpose(1, 2)
            if feat.shape[-1] == self.feature_dim:
                feat = feat.permute(0, 3, 1, 2)
                feat = self.cnn_proj(feat)
                return feat.flatten(2).transpose(1, 2)
        if feat.ndim == 3:
            if feat.shape[-1] == self.feature_dim:
                return self.token_proj(feat)
            if feat.shape[1] == self.feature_dim:
                return self.token_proj(feat.transpose(1, 2))
        raise ValueError(f"Unsupported feature shape: {tuple(feat.shape)} for backbone={self.config.backbone_name}")

    def encode_views(self, views):
        f1 = self.backbone.forward_features(views[0])
        f2 = self.backbone.forward_features(views[1])
        t1 = self._to_tokens(f1)
        t2 = self._to_tokens(f2)
        for blk12, blk21 in zip(self.cross_12, self.cross_21):
            t1_prev, t2_prev = t1, t2
            t1 = blk12(t1_prev, t2_prev)
            t2 = blk21(t2_prev, t1_prev)
        return torch.cat([t1.mean(dim=1), t2.mean(dim=1)], dim=1)

    def forward(self, views):
        feat = self.encode_views(views)
        return self.classifier(feat)


In [3]:
def _center_crop_and_resize(x, crop_ratio=0.95):
    b, c, h, w = x.shape
    ch, cw = int(h * crop_ratio), int(w * crop_ratio)
    y1 = (h - ch) // 2
    x1 = (w - cw) // 2
    cropped = x[:, :, y1:y1 + ch, x1:x1 + cw]
    return F.interpolate(cropped, size=(h, w), mode="bilinear", align_corners=False)


def apply_tta_to_views(views, tta_name):
    if tta_name == "none":
        return views
    if tta_name == "hflip":
        return [torch.flip(v, dims=[3]) for v in views]
    if tta_name == "crop95":
        return [_center_crop_and_resize(v, crop_ratio=0.95) for v in views]
    raise ValueError(f"Unknown TTA: {tta_name}")


def logloss_from_probs(labels, probs):
    probs = np.clip(np.asarray(probs, dtype=np.float64), 1e-15, 1 - 1e-15)
    labels = np.asarray(labels, dtype=np.float64)
    return float(-np.mean(labels * np.log(probs) + (1 - labels) * np.log(1 - probs)))


def predict_probs_with_tta(model, loader, device, tta_names=None, desc="Inference TTA"):
    tta_names = tta_names or ["none"]
    model.eval()
    all_probs, all_labels, all_ids = [], [], []
    with torch.no_grad():
        for views, labels in tqdm(loader, desc=desc, dynamic_ncols=True, ascii=True):
            all_labels.extend(labels.numpy())
            views = [v.to(device) for v in views]
            probs_sum = None
            for tta_name in tta_names:
                tta_views = apply_tta_to_views(views, tta_name)
                logits = model(tta_views).view(-1)
                probs = torch.sigmoid(logits)
                probs_sum = probs if probs_sum is None else (probs_sum + probs)
            all_probs.extend((probs_sum / len(tta_names)).cpu().numpy())
    return np.array(all_probs, dtype=np.float64), np.array(all_labels, dtype=np.float64)


def extract_embeddings(model, loader, device, desc="Extract Features"):
    model.eval()
    all_feats, all_labels = [], []
    with torch.no_grad():
        for views, labels in tqdm(loader, desc=desc, dynamic_ncols=True, ascii=True):
            views = [v.to(device) for v in views]
            feats = model.encode_views(views)
            all_feats.append(feats.cpu().numpy())
            all_labels.extend(labels.numpy())
    return np.concatenate(all_feats, axis=0), np.array(all_labels, dtype=np.float64)


def linear_cka(x: np.ndarray, y: np.ndarray) -> float:
    x = x - x.mean(axis=0, keepdims=True)
    y = y - y.mean(axis=0, keepdims=True)
    hsic = np.linalg.norm(x.T @ y, ord="fro") ** 2
    x_norm = np.linalg.norm(x.T @ x, ord="fro")
    y_norm = np.linalg.norm(y.T @ y, ord="fro")
    denom = x_norm * y_norm
    if denom == 0:
        return 0.0
    return float(hsic / denom)


def plot_matrix(df: pd.DataFrame, title: str, save_path: Path, cmap: str = "viridis", fmt: str = ".3f"):
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(df.to_numpy(), cmap=cmap)
    ax.set_xticks(range(len(df.columns)))
    ax.set_xticklabels(df.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(df.index)))
    ax.set_yticklabels(df.index)
    ax.set_title(title)
    for i in range(df.shape[0]):
        for j in range(df.shape[1]):
            ax.text(j, i, format(df.iloc[i, j], fmt), ha="center", va="center", color="white" if abs(df.iloc[i, j]) > 0.5 else "black", fontsize=8)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(save_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


In [4]:
eval_df = pd.read_csv(EVAL_CSV)
eval_df["best_tta"] = eval_df["best_tta"].apply(ast.literal_eval)
display(eval_df[["backbone", "best_val_logloss", "tta_logloss", "tta_acc", "best_tta"]])

train_transform, test_transform = build_default_transforms(CFG["IMG_SIZE"])
val_df = pd.read_csv(DATA_DIR / "dev.csv", encoding="utf-8-sig")
val_df["sample_dir"] = str(DATA_DIR / "dev")
val_dataset = MultiViewDataset(val_df, str(DATA_DIR / "dev"), test_transform, is_test=False)
val_loader = DataLoader(
    val_dataset,
    batch_size=CFG["BATCH_SIZE"],
    shuffle=False,
    num_workers=CFG["NUM_WORKERS"],
    pin_memory=(device.type == "cuda"),
    worker_init_fn=seed_worker,
    generator=make_generator(CFG["SEED"] + 1),
)
print("val rows:", len(val_dataset))


def resolve_checkpoint_path(path_str: str) -> Path:
    path = Path(path_str)
    if path.exists():
        return path
    fallback = ROOT / "outputs" / "weights" / "mv_caf_backbone_test" / "v1.2" / path.name
    if fallback.exists():
        return fallback
    raise FileNotFoundError(f"Checkpoint not found: {path_str} (fallback tried: {fallback})")


,backbone,best_val_logloss,tta_logloss,tta_acc,best_tta
0,convnext_small.fb_in22k_ft_in1k,0.097624,0.097624,0.98,[none]
1,swin_tiny_patch4_window7_224.ms_in22k_ft_in1k,0.143009,0.111031,0.96,"[none, hflip]"
2,convnext_tiny.fb_in22k_ft_in1k,0.116654,0.113805,0.95,"[none, hflip, crop95]"
3,deit3_small_patch16_224.fb_in22k_ft_in1k,0.139015,0.121513,0.96,"[none, hflip]"
4,efficientnetv2_rw_s,0.250585,0.250585,0.88,[none]
5,vit_small_patch16_224.augreg_in21k_ft_in1k,0.318323,0.318323,0.89,[none]
6,vit_base_patch16_224.augreg_in21k_ft_in1k,0.549145,0.549145,0.80,[none]


val rows: 100


In [5]:
prediction_rows = []
feature_bank = {}
labels_ref = None

for row in eval_df.itertuples(index=False):
    backbone_name = row.backbone
    checkpoint_path = resolve_checkpoint_path(row.checkpoint_path)
    tta_names = row.best_tta

    model = MultiViewFlexibleCAF(FlexibleCAFConfig(backbone_name=backbone_name)).to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    state_dict = checkpoint["ema_state_dict"] if "ema_state_dict" in checkpoint else checkpoint["model_state_dict"]
    model.load_state_dict(state_dict)

    probs, labels = predict_probs_with_tta(model, val_loader, device, tta_names=tta_names, desc=f"Dev probs {backbone_name}")
    feats, feat_labels = extract_embeddings(model, val_loader, device, desc=f"Features {backbone_name}")

    if labels_ref is None:
        labels_ref = labels
    else:
        assert np.array_equal(labels_ref, labels)
        assert np.array_equal(labels_ref, feat_labels)

    feature_bank[backbone_name] = feats
    prediction_rows.append(pd.DataFrame({
        "idx": np.arange(len(labels)),
        "label": labels,
        "backbone": backbone_name,
        "prob": probs,
        "pred": (probs > 0.5).astype(int),
        "error": ((probs > 0.5).astype(int) != labels.astype(int)).astype(int),
    }))

pred_long_df = pd.concat(prediction_rows, ignore_index=True)
pred_wide_df = pred_long_df.pivot(index="idx", columns="backbone", values="prob").reset_index()
pred_label_df = pred_long_df[["idx", "label"]].drop_duplicates().sort_values("idx")
pred_wide_df = pred_label_df.merge(pred_wide_df, on="idx", how="left")
display(pred_wide_df.head())

pred_long_df.to_csv(OUT_DIR / "dev_predictions_long.csv", index=False)
pred_wide_df.to_csv(OUT_DIR / "dev_predictions_wide.csv", index=False)


Features convnext_small.fb_in22k_ft_in1k: 100%|##########| 7/7 [00:01<00:00,  6.99it/s]
Dev probs swin_tiny_patch4_window7_224.ms_in22k_ft_in1k: 100%|##########| 7/7 [00:01<00:00,  4.74it/s]
Features swin_tiny_patch4_window7_224.ms_in22k_ft_in1k: 100%|##########| 7/7 [00:00<00:00,  7.66it/s]
Features convnext_tiny.fb_in22k_ft_in1k: 100%|##########| 7/7 [00:00<00:00,  9.17it/s]
Dev probs deit3_small_patch16_224.fb_in22k_ft_in1k: 100%|##########| 7/7 [00:01<00:00,  6.55it/s]
Features deit3_small_patch16_224.fb_in22k_ft_in1k: 100%|##########| 7/7 [00:00<00:00,  7.84it/s]
Features efficientnetv2_rw_s: 100%|##########| 7/7 [00:00<00:00,  7.95it/s]
Dev probs vit_small_patch16_224.augreg_in21k_ft_in1k: 100%|##########| 7/7 [00:00<00:00, 10.16it/s]
Features vit_small_patch16_224.augreg_in21k_ft_in1k: 100%|##########| 7/7 [00:00<00:00,  9.84it/s]
Dev probs vit_base_patch16_224.augreg_in21k_ft_in1k: 100%|##########| 7/7 [00:01<00:00,  5.30it/s]
Features vit_base_patch16_224.augreg_in21k_ft_in1k:

,idx,label,convnext_small.fb_in22k_ft_in1k,convnext_tiny.fb_in22k_ft_in1k,deit3_small_patch16_224.fb_in22k_ft_in1k,efficientnetv2_rw_s,swin_tiny_patch4_window7_224.ms_in22k_ft_in1k,vit_base_patch16_224.augreg_in21k_ft_in1k,vit_small_patch16_224.augreg_in21k_ft_in1k
0,0,1.0,0.931487,0.997509,0.968336,0.968959,0.971802,0.608289,0.899366
1,1,1.0,0.994949,0.999155,0.996415,0.987314,0.988716,0.617065,0.939289
2,2,1.0,0.985100,0.998102,0.994832,0.968576,0.976252,0.616912,0.939395
3,3,0.0,0.015393,0.016148,0.114207,0.190261,0.095174,0.429184,0.833207
4,4,0.0,0.064896,0.005942,0.125574,0.612286,0.202964,0.508586,0.679178


In [6]:
backbones = eval_df["backbone"].tolist()

corr_mat = pd.DataFrame(index=backbones, columns=backbones, dtype=float)
disagree_mat = pd.DataFrame(index=backbones, columns=backbones, dtype=float)
error_overlap_mat = pd.DataFrame(index=backbones, columns=backbones, dtype=float)
ensemble_gain_mat = pd.DataFrame(index=backbones, columns=backbones, dtype=float)
cka_mat = pd.DataFrame(index=backbones, columns=backbones, dtype=float)

pair_rows = []
labels = labels_ref.astype(int)

for a in backbones:
    probs_a = pred_wide_df[a].to_numpy(dtype=np.float64)
    pred_a = (probs_a > 0.5).astype(int)
    err_a = pred_a != labels
    ll_a = logloss_from_probs(labels, probs_a)
    for b in backbones:
        probs_b = pred_wide_df[b].to_numpy(dtype=np.float64)
        pred_b = (probs_b > 0.5).astype(int)
        err_b = pred_b != labels
        ll_b = logloss_from_probs(labels, probs_b)
        ensemble_probs = 0.5 * (probs_a + probs_b)
        ensemble_ll = logloss_from_probs(labels, ensemble_probs)

        corr = float(np.corrcoef(probs_a, probs_b)[0, 1]) if a != b else 1.0
        disagree = float(np.mean(pred_a != pred_b)) if a != b else 0.0
        union_err = np.logical_or(err_a, err_b)
        error_overlap = float(np.logical_and(err_a, err_b).sum() / max(union_err.sum(), 1))
        gain = float(min(ll_a, ll_b) - ensemble_ll)
        cka = linear_cka(feature_bank[a], feature_bank[b]) if a != b else 1.0

        corr_mat.loc[a, b] = corr
        disagree_mat.loc[a, b] = disagree
        error_overlap_mat.loc[a, b] = error_overlap
        ensemble_gain_mat.loc[a, b] = gain
        cka_mat.loc[a, b] = cka

for a, b in combinations(backbones, 2):
    probs_a = pred_wide_df[a].to_numpy(dtype=np.float64)
    probs_b = pred_wide_df[b].to_numpy(dtype=np.float64)
    pred_a = (probs_a > 0.5).astype(int)
    pred_b = (probs_b > 0.5).astype(int)
    err_a = pred_a != labels
    err_b = pred_b != labels
    ensemble_probs = 0.5 * (probs_a + probs_b)
    ensemble_ll = logloss_from_probs(labels, ensemble_probs)
    pair_rows.append({
        "model_a": a,
        "model_b": b,
        "logloss_a": logloss_from_probs(labels, probs_a),
        "logloss_b": logloss_from_probs(labels, probs_b),
        "best_single_logloss": min(logloss_from_probs(labels, probs_a), logloss_from_probs(labels, probs_b)),
        "ensemble_logloss": ensemble_ll,
        "ensemble_gain": min(logloss_from_probs(labels, probs_a), logloss_from_probs(labels, probs_b)) - ensemble_ll,
        "prob_corr": float(np.corrcoef(probs_a, probs_b)[0, 1]),
        "prediction_disagreement": float(np.mean(pred_a != pred_b)),
        "both_wrong": int(np.logical_and(err_a, err_b).sum()),
        "only_a_wrong": int(np.logical_and(err_a, ~err_b).sum()),
        "only_b_wrong": int(np.logical_and(~err_a, err_b).sum()),
        "either_wrong": int(np.logical_or(err_a, err_b).sum()),
        "error_overlap_ratio": float(np.logical_and(err_a, err_b).sum() / max(np.logical_or(err_a, err_b).sum(), 1)),
        "feature_cka": linear_cka(feature_bank[a], feature_bank[b]),
    })

pair_df = pd.DataFrame(pair_rows).sort_values(["ensemble_gain", "prediction_disagreement"], ascending=[False, False]).reset_index(drop=True)
display(pair_df.head(10))

pair_df.to_csv(OUT_DIR / "pairwise_complementarity.csv", index=False)
corr_mat.to_csv(OUT_DIR / "pairwise_prob_corr.csv")
disagree_mat.to_csv(OUT_DIR / "pairwise_prediction_disagreement.csv")
error_overlap_mat.to_csv(OUT_DIR / "pairwise_error_overlap.csv")
ensemble_gain_mat.to_csv(OUT_DIR / "pairwise_ensemble_gain.csv")
cka_mat.to_csv(OUT_DIR / "pairwise_feature_cka.csv")


,model_a,model_b,logloss_a,logloss_b,best_single_logloss,ensemble_logloss,ensemble_gain,prob_corr,prediction_disagreement,both_wrong,only_a_wrong,only_b_wrong,either_wrong,error_overlap_ratio,feature_cka
0,swin_tiny_patch4_window7_224.ms_in22k_ft_in1k,convnext_tiny.fb_in22k_ft_in1k,0.111031,0.113805,0.111031,0.096802,0.014229,0.918496,0.05,2,2,3,7,0.285714,0.851026
1,convnext_tiny.fb_in22k_ft_in1k,deit3_small_patch16_224.fb_in22k_ft_in1k,0.113805,0.121513,0.113805,0.105454,0.008351,0.946866,0.05,2,3,2,7,0.285714,0.893338
2,convnext_small.fb_in22k_ft_in1k,swin_tiny_patch4_window7_224.ms_in22k_ft_in1k,0.097624,0.111031,0.097624,0.090212,0.007412,0.944418,0.04,1,1,3,5,0.200000,0.888415
3,swin_tiny_patch4_window7_224.ms_in22k_ft_in1k,deit3_small_patch16_224.fb_in22k_ft_in1k,0.111031,0.121513,0.111031,0.106048,0.004983,0.936654,0.06,1,3,3,7,0.142857,0.860594
4,convnext_small.fb_in22k_ft_in1k,deit3_small_patch16_224.fb_in22k_ft_in1k,0.097624,0.121513,0.097624,0.093944,0.003680,0.935808,0.06,0,2,4,6,0.000000,0.887577
5,convnext_small.fb_in22k_ft_in1k,convnext_tiny.fb_in22k_ft_in1k,0.097624,0.113805,0.097624,0.097103,0.000521,0.964912,0.03,2,0,3,5,0.400000,0.932111
6,efficientnetv2_rw_s,vit_small_patch16_224.augreg_in21k_ft_in1k,0.250585,0.318323,0.250585,0.257718,-0.007133,0.842490,0.11,6,6,5,17,0.352941,0.672035
7,deit3_small_patch16_224.fb_in22k_ft_in1k,efficientnetv2_rw_s,0.121513,0.250585,0.121513,0.152888,-0.031375,0.823782,0.16,0,4,12,16,0.000000,0.730717
8,convnext_tiny.fb_in22k_ft_in1k,efficientnetv2_rw_s,0.113805,0.250585,0.113805,0.149827,-0.036021,0.861267,0.13,2,3,10,15,0.133333,0.780748
9,swin_tiny_patch4_window7_224.ms_in22k_ft_in1k,efficientnetv2_rw_s,0.111031,0.250585,0.111031,0.151349,-0.040318,0.851863,0.14,1,3,11,15,0.066667,0.770139


In [7]:
plot_matrix(corr_mat, "Prediction Correlation (dev probs)", OUT_DIR / "prob_corr_heatmap.png", cmap="magma")
plot_matrix(disagree_mat, "Prediction Disagreement Rate", OUT_DIR / "prediction_disagreement_heatmap.png", cmap="viridis")
plot_matrix(error_overlap_mat, "Error Overlap Ratio", OUT_DIR / "error_overlap_heatmap.png", cmap="cividis")
plot_matrix(ensemble_gain_mat, "Pairwise Ensemble Gain (best single - ensemble logloss)", OUT_DIR / "ensemble_gain_heatmap.png", cmap="coolwarm")
plot_matrix(cka_mat, "Feature CKA", OUT_DIR / "feature_cka_heatmap.png", cmap="plasma")

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(pair_df["prob_corr"], pair_df["ensemble_gain"], alpha=0.8)
for row in pair_df.head(12).itertuples(index=False):
    label = f"{row.model_a.split('.')[0]}\n+ {row.model_b.split('.')[0]}"
    ax.annotate(label, (row.prob_corr, row.ensemble_gain), fontsize=7)
ax.set_xlabel("Prediction correlation")
ax.set_ylabel("Ensemble gain")
ax.set_title("Lower correlation can create ensemble gain")
fig.tight_layout()
fig.savefig(OUT_DIR / "correlation_vs_ensemble_gain.png", dpi=180, bbox_inches="tight")
plt.close(fig)

print("saved plots to:", OUT_DIR)


saved plots to: /media/hdd0/whyz/structure-stability/outputs/eda_preprocessing/ensemble_sampling/v1.2


In [ ]:
summary_df = pair_df[[
    "model_a", "model_b", "best_single_logloss", "ensemble_logloss", "ensemble_gain",
    "prob_corr", "prediction_disagreement", "error_overlap_ratio", "feature_cka"
]].copy()
summary_df = summary_df.sort_values(["ensemble_gain", "prediction_disagreement"], ascending=[False, False]).reset_index(drop=True)
display(summary_df.head(15))

positive_gain = summary_df[summary_df["ensemble_gain"] > 0]
print({
    "num_models": int(len(backbones)),
    "num_pairs": int(len(summary_df)),
    "pairs_with_positive_gain": int(len(positive_gain)),
    "best_pair": positive_gain.iloc[0][["model_a", "model_b", "ensemble_gain"]].to_dict() if len(positive_gain) else None,
})


,model_a,model_b,best_single_logloss,ensemble_logloss,ensemble_gain,prob_corr,prediction_disagreement,error_overlap_ratio,feature_cka
0,swin_tiny_patch4_window7_224.ms_in22k_ft_in1k,convnext_tiny.fb_in22k_ft_in1k,0.111031,0.096802,0.014229,0.918496,0.05,0.285714,0.851026
1,convnext_tiny.fb_in22k_ft_in1k,deit3_small_patch16_224.fb_in22k_ft_in1k,0.113805,0.105454,0.008351,0.946866,0.05,0.285714,0.893338
2,convnext_small.fb_in22k_ft_in1k,swin_tiny_patch4_window7_224.ms_in22k_ft_in1k,0.097624,0.090212,0.007412,0.944418,0.04,0.200000,0.888415
3,swin_tiny_patch4_window7_224.ms_in22k_ft_in1k,deit3_small_patch16_224.fb_in22k_ft_in1k,0.111031,0.106048,0.004983,0.936654,0.06,0.142857,0.860594
4,convnext_small.fb_in22k_ft_in1k,deit3_small_patch16_224.fb_in22k_ft_in1k,0.097624,0.093944,0.003680,0.935808,0.06,0.000000,0.887577
5,convnext_small.fb_in22k_ft_in1k,convnext_tiny.fb_in22k_ft_in1k,0.097624,0.097103,0.000521,0.964912,0.03,0.400000,0.932111
6,efficientnetv2_rw_s,vit_small_patch16_224.augreg_in21k_ft_in1k,0.250585,0.257718,-0.007133,0.842490,0.11,0.352941,0.672035
7,deit3_small_patch16_224.fb_in22k_ft_in1k,efficientnetv2_rw_s,0.121513,0.152888,-0.031375,0.823782,0.16,0.000000,0.730717
8,convnext_tiny.fb_in22k_ft_in1k,efficientnetv2_rw_s,0.113805,0.149827,-0.036021,0.861267,0.13,0.133333,0.780748
9,swin_tiny_patch4_window7_224.ms_in22k_ft_in1k,efficientnetv2_rw_s,0.111031,0.151349,-0.040318,0.851863,0.14,0.066667,0.770139


{'num_models': 7, 'num_pairs': 21, 'pairs_with_positive_gain': 6, 'best_pair': {'model_a': 'swin_tiny_patch4_window7_224.ms_in22k_ft_in1k', 'model_b': 'convnext_tiny.fb_in22k_ft_in1k', 'ensemble_gain': 0.014228995524544832}}


: 